# 강의 04 · 실습 1 — 상태·노드·조건부 엣지 · (2-2) 빈칸 채우기 II

## 1. 문제상황

- 온라인 쇼핑몰 고객센터는 하루에 여러 통의 메일을 받습니다.
- 받은 메일에는 환불 문의와 배송 문의가 들어 있고, 그 사이에 광고 메일이 섞여 들어옵니다.
- 담당자는 메일 한 통마다 종류를 눈으로 판단하고, 광고 메일은 버리고, 나머지 메일에는 답장을 직접 씁니다.
- 메일 수가 늘어나면 이 판단과 작성을 사람이 그만큼 반복해야 합니다.

## 2. 문제와 목표

- **문제**: 메일의 종류를 사람이 판단하고, 답장 초안도 사람이 씁니다. 두 가지 일이 메일 수만큼 반복됩니다.
- **목표**
  - 메일 한 통을 입력하면 프로그램이 종류를 판단합니다.
    - 종류 셋: 환불, 배송, 스팸
  - 광고 메일이면 그 자리에서 처리를 끝내고, 광고 메일이 아니면 답장 초안을 만들어 발송까지 진행하는 처리 흐름을 만듭니다.
- **목표 달성 여부의 판정 기준**:
  - 광고 메일 한 통과 환불 문의 한 통을 입력했을 때,
  - 광고 메일은 분류 단계에서 끝나고 환불 문의는 답장 초안 작성과 발송까지 진행되는 것을 실행 결과에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec04_ex01_s1_diagram.svg)

## 4. 단계별 요구사항

1. **상태를 정의합니다.**
    - 메일 본문(`email`), 분류 결과(`category`), 답장 초안(`draft`), 발송 여부(`sent`) 키 네 개를 가지는 상태를 선언합니다.
    - 키 네 개 외의 값은 상태에 들어가지 않습니다.
2. **분류 노드를 만듭니다.**
    - classify 노드는 상태의 메일 본문을 읽고, 환불·배송·스팸 중 한 단어로 분류한 결과를 상태의 `category` 키에 씁니다.
3. **답장 노드를 만듭니다.**
    - draft 노드는 상태의 분류 결과와 메일 본문을 읽고, 두 문장짜리 답장 초안을 상태의 `draft` 키에 씁니다.
4. **발송 노드를 만듭니다.**
    - send 노드는 상태의 답장 초안을 발송하고, 상태의 `sent` 키에 발송 여부를 씁니다.
    - 이 실습에서 발송은 화면 출력으로 대신합니다.
5. **그래프에 노드를 등록합니다.**
    - 세 노드를 이름과 함께 그래프에 등록합니다.
6. **엣지를 연결합니다.**
    - START에서 classify로 가는 고정 엣지를 추가하고, classify 뒤에는 분류 결과를 보고 갈 곳을 고르는 조건부 엣지를 추가합니다.
    - 분류 결과가 스팸이면 END로 가고, 스팸이 아니면 draft로 갑니다.
    - draft 뒤에는 send를, send 뒤에는 END를 고정 엣지로 연결합니다.
7. **그래프를 컴파일하고 실행합니다.**
    - 환불 문의 메일과 광고 메일을 차례대로 넣고, 노드가 하나 끝날 때마다 상태의 어느 키가 채워졌는지 화면에 출력합니다.

## 5. 코드 골격 — LangGraph 5단

랭그래프(LangGraph)로 그래프를 세우는 순서는 다음 다섯 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 다섯 단계와 하나씩 대응합니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 상태 정의 | 노드들이 함께 읽고 쓸 키를 선언합니다 | `class EmailState(TypedDict)` | 1 |
| ② 노드 함수 정의 | 상태를 받아 바뀐 키만 돌려주는 함수를 만듭니다 | `def classify(state) -> dict` | 2, 3, 4 |
| ③ 그래프 빌더 생성과 노드 등록 | 빈 그래프를 열고 함수에 이름을 붙여 등록합니다 | `StateGraph(EmailState)`, `add_node` | 5 |
| ④ 엣지 연결 | 노드 사이의 순서와 분기를 정합니다 | `add_edge`, `add_conditional_edges` | 6 |
| ⑤ 컴파일과 실행 | 연결을 확정하고 입력을 넣어 실행합니다 | `compile()`, `stream()` | 7 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
# 여기에 단계 0(라이브러리 불러오기, .env 읽기, 모델 준비)을 작성합니다.

### 단계 ① — 상태 정의 (요구사항 1)

그래프가 도는 동안 모든 노드가 함께 읽고 쓰는 키를 선언합니다. `TypedDict`로 선언한 네 개의 키가 이 그래프에서 오가는 데이터의 전부입니다.

In [ ]:
# 여기에 단계 ①(상태 정의)을 작성합니다.

### 단계 ② — 노드 함수 정의 (요구사항 2, 3, 4)

- 노드는 상태를 인자로 받아 딕셔너리를 돌려주는 파이썬 함수입니다.
- 돌려준 딕셔너리가 상태의 해당 키를 덮습니다. 바뀐 키만 돌려주면 나머지 키는 그대로 남습니다.
- 노드 안의 `SystemMessage`는 그 노드가 모델을 한 번 호출할 때 쓰는 시스템 프롬프트입니다. `HumanMessage`는 사용자 입력에 해당합니다. 노드마다 시스템 프롬프트가 따로 있습니다.

In [ ]:
# 여기에 단계 ②(노드 함수 세 개 정의)를 작성합니다.

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 5)

`StateGraph`에 상태를 넘겨 빈 그래프를 열고, `add_node`로 함수마다 이름을 붙여 등록합니다. 여기서 붙인 이름은 뒤의 엣지 연결에서 그대로 쓰입니다.

In [ ]:
# 여기에 단계 ③(그래프 빌더 생성과 노드 등록)을 작성합니다.

### 단계 ④ — 엣지 연결 (요구사항 6)

`add_edge`는 고정된 순서로 연결합니다. `add_conditional_edges`는 판단 함수가 돌려준 이름으로 다음 노드를 고르는 분기를 추가합니다. 판단 함수 `route`는 상태의 분류 결과만 보고 갈 곳의 이름을 돌려줍니다. 돌려주는 이름은 그래프에 등록된 노드 이름이거나 `END`여야 합니다.

In [ ]:
# 여기에 단계 ④(엣지 연결)를 작성합니다.

### 단계 ⑤ — 컴파일과 실행 (요구사항 7)

`compile()`이 연결을 확정해 실행 가능한 그래프를 돌려줍니다. `stream`은 노드가 하나 끝날 때마다 그 노드가 바꾼 부분을 내보냅니다. 아래에서는 환불 문의 메일과 광고 메일을 차례대로 넣습니다.

In [ ]:
# 여기에 단계 ⑤(컴파일과 실행)를 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. 1번 입력(환불 문의)에서는 `classify`, `draft`, `send` 세 노드가 차례대로 출력됩니다. 각 노드가 상태의 `category`, `draft`, `sent` 키를 하나씩 채웁니다.
2. 2번 입력(광고 메일)에서는 `classify` 한 줄만 출력됩니다. 분류 결과가 스팸이므로 조건부 엣지가 그래프를 END로 보냈고, `draft`와 `send`는 실행되지 않습니다.
3. 두 입력의 마지막 줄에서 `sent` 값이 다릅니다. 환불 문의는 `True`이고, 광고 메일은 `None`입니다. `None`은 그 키를 채운 노드가 없었다는 뜻입니다.

세 가지가 모두 확인되면 완성입니다. 하나라도 다르면 `lec04_ex01_s1.ipynb`와 대조해 채운 빈칸을 고칩니다.